In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

if os.environ["OPENAI_API_KEY"]:
    print("API key is set.")

API key is set.


In [1]:
from pathlib import Path

print("Current working directory:")
print(Path.cwd())

print("\nDocs folder:")
print(Path("./docs").resolve())

print("\nVector folder:")
print(Path("./vector").resolve())

Current working directory:
/Users/sumi/Documents/All Projects/3.Rag_application

Docs folder:
/Users/sumi/Documents/All Projects/3.Rag_application/docs

Vector folder:
/Users/sumi/Documents/All Projects/3.Rag_application/vector


In [2]:
from pathlib import Path

vector_directory = Path("./vector")

print("Vector folder exists:", vector_directory.exists())

Vector folder exists: False


In [4]:
from langchain_openai import ChatOpenAI

In [5]:
llm = ChatOpenAI(model="gpt-5-nano", temperature = 0)


# Rag implemantation with pdf

#### **Step 1: Extracting Text from pdf**

In [6]:
# from langchain_community.document_loaders import PyPDFLoader
# # pdf_path = "./docs/fabric-admin.pdf"
# pdf_path = "./docs/Pharmacology-WEB.pdf"
# loader = PyPDFLoader(pdf_path)
# docs = loader.load()
# docs

# Load every pdf in docs folder
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

loader = DirectoryLoader(
    path="./docs",
    glob="*.pdf",
    loader_cls=PyPDFLoader,
    show_progress=True
)

docs = loader.load()

print(f"Total PDF pages loaded: {len(docs)}")

/var/folders/ky/lw33txtn4fsdnh2zmn9r1yt80000gn/T/ipykernel_40460/1689989629.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
100%|██████████| 3/3 [00:13<00:00,  4.37s/it]

Total PDF pages loaded: 2409


### **Creating own metadata for pdf chunks**

In [7]:
# for i in docs:
#     i.metadata = {"source" : "fabric-admin.pdf","developer": "Microsoft"}
for doc in docs:
    doc.metadata["domain"] = "Pharmacology"
    doc.metadata["language"] = "English"

In [8]:
docs[0].metadata

{'producer': 'Acrobat Distiller 8.1.0 (Macintosh)',
 'creator': 'Adobe InDesign CS4 (6.0.4)',
 'creationdate': '2010-03-08T16:51:34+13:00',
 'author': 'Danish, Fazal-I-Akbar(Author)',
 'keywords': '',
 'moddate': '2016-06-08T10:26:37+05:30',
 'title': 'MasterPass : Pharmacology in 7 Days for Medical Students',
 'ebx_publisher': '/CRC Press',
 'source': 'docs/Pharmacology 7 days.pdf',
 'total_pages': 160,
 'page': 0,
 'page_label': 'A',
 'domain': 'Pharmacology',
 'language': 'English'}

#### **Step 2: Splitting the Documents into CHUNKS**

In [9]:

from langchain_core.documents import Document


In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

chunks = splitter.split_documents(docs)

chunks

[Document(metadata={'producer': 'Acrobat Distiller 8.1.0 (Macintosh)', 'creator': 'Adobe InDesign CS4 (6.0.4)', 'creationdate': '2010-03-08T16:51:34+13:00', 'author': 'Danish, Fazal-I-Akbar(Author)', 'keywords': '', 'moddate': '2016-06-08T10:26:37+05:30', 'title': 'MasterPass : Pharmacology in 7 Days for Medical Students', 'ebx_publisher': '/CRC Press', 'source': 'docs/Pharmacology 7 days.pdf', 'total_pages': 160, 'page': 1, 'page_label': 'i', 'domain': 'Pharmacology', 'language': 'English'}, page_content='Pharmacology \nin 7 Days for \nMedical Students\nFAZAL-I-AKBAR DANISH\nCT2 in Medicine\nPrincess of Wales Hospital in Bridgend\nand\nAHMED EHSAN RABBANI\nFinal Y ear Medical Student\nFoundation University Medical College (FUMC)\nRawalpindi, Pakistan\nRadcliffe Publishing\nOxford • New Y ork'),
 Document(metadata={'producer': 'Acrobat Distiller 8.1.0 (Macintosh)', 'creator': 'Adobe InDesign CS4 (6.0.4)', 'creationdate': '2010-03-08T16:51:34+13:00', 'author': 'Danish, Fazal-I-Akbar(Aut

In [11]:
len(chunks)

9176

In [12]:
chunks[0].metadata

{'producer': 'Acrobat Distiller 8.1.0 (Macintosh)',
 'creator': 'Adobe InDesign CS4 (6.0.4)',
 'creationdate': '2010-03-08T16:51:34+13:00',
 'author': 'Danish, Fazal-I-Akbar(Author)',
 'keywords': '',
 'moddate': '2016-06-08T10:26:37+05:30',
 'title': 'MasterPass : Pharmacology in 7 Days for Medical Students',
 'ebx_publisher': '/CRC Press',
 'source': 'docs/Pharmacology 7 days.pdf',
 'total_pages': 160,
 'page': 1,
 'page_label': 'i',
 'domain': 'Pharmacology',
 'language': 'English'}

#### **Step 3: Creating Embedding for the CHUNKS**

In [16]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma


# Create the embedding model.
# This converts every text chunk into a numerical vector.
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small"
)


# Create a new local Chroma vector database.
# The database will be saved inside the project-level vector folder.
vectorstore = Chroma(
    collection_name="pharmacology_books",
    persist_directory="./vector",
    embedding_function=embedding_model
)


# A newly created database should initially contain zero chunks.
print("Current stored chunks:", vectorstore._collection.count())

Current stored chunks: 0


In [17]:
from tqdm.auto import tqdm


# Insert 250 chunks at a time.
# This avoids exceeding Chroma's batch-size limit
# and reduces pressure on the local vector index.
batch_size = 250


# Move through the complete list of chunks in groups of 250.
for start_index in tqdm(
    range(0, len(chunks), batch_size),
    desc="Adding chunks to Chroma"
):
    # Calculate the ending position of the current batch.
    end_index = min(
        start_index + batch_size,
        len(chunks)
    )

    # Select the chunks for the current batch.
    current_batch = chunks[start_index:end_index]

    # Generate embeddings and store:
    # - vector embeddings
    # - original chunk text
    # - metadata such as source and page
    vectorstore.add_documents(current_batch)


print("All chunks were stored successfully.")

/Users/sumi/Documents/All Projects/3.Rag_application/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Adding chunks to Chroma: 100%|██████████| 37/37 [01:03<00:00,  1.72s/it]

All chunks were stored successfully.


In [18]:
stored_chunk_count = vectorstore._collection.count()

print(f"Expected chunks: {len(chunks)}")
print(f"Stored chunks:   {stored_chunk_count}")

Expected chunks: 9176
Stored chunks:   9176


#### **Step 4: Store embedding in the existing local  vector store**

In [ ]:
# from langchain_community.vectorstores import Chroma
# vectorstore = Chroma(
#     persist_directory = "./vector/", 
#     embedding_function = embedding_model)


/var/folders/ky/lw33txtn4fsdnh2zmn9r1yt80000gn/T/ipykernel_31847/1267599070.py:2: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


In [20]:
# Add our embeddings to the vectorstore
# vectorstore.add_documents(chunks)

#### **Step 5: Similarity search/ Semantic Search**

In [21]:
vectorstore.similarity_search("what is Pharmacology?", k=3)

[Document(id='65455062-8d20-4fb3-a79f-051ad7b3d4b2', metadata={'domain': 'Pharmacology', 'producer': 'Acrobat Distiller 9.3.0 (Windows)', 'moddate': '2014-12-26T00:39:05+01:00', 'creationdate': '2014-12-25T23:25:36+01:00', 'source': 'docs/medical-pharmacology.pdf', 'creator': 'PageMaker 7.0', 'page_label': '2', 'total_pages': 1020, 'language': 'English', 'page': 18}, page_content='The term ‘drugs’ is being also used to mean\naddictive/abused/illicit substances. However, this\nrestricted and derogatory sense usage is unfor-\ntunate degradation of a time honoured term, and\n‘drug’ should refer to a substance that has some\ntherapeutic/diagnostic application.\nSome other important aspects of pharmacology\nare:\nPharmacotherapeutics It is the application\nof pharmacological information together with\nknowledge of the disease for its prevention,\nmitigation or cure. Selection of the most appro-\npriate drug, dosage and duration of treatment\ntaking into account the specific features of a\np

#### **Reuse the vector database**

In [22]:
import os
from dotenv import load_dotenv
load_dotenv()

if os.environ["OPENAI_API_KEY"]:
    print("API key is set.")



API key is set.


In [23]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-5-nano", temperature = 0)




In [24]:
from langchain_openai import OpenAIEmbeddings
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
from langchain_community.vectorstores import Chroma
vectorstore_persist = Chroma(
    persist_directory = "./vector/",
    embedding_function = embedding_model)

/var/folders/ky/lw33txtn4fsdnh2zmn9r1yt80000gn/T/ipykernel_40460/1158764723.py:4: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore_persist = Chroma(


In [25]:
vectorstore.similarity_search("what is Pharmacology?", k=3)

[Document(id='65455062-8d20-4fb3-a79f-051ad7b3d4b2', metadata={'creationdate': '2014-12-25T23:25:36+01:00', 'moddate': '2014-12-26T00:39:05+01:00', 'producer': 'Acrobat Distiller 9.3.0 (Windows)', 'total_pages': 1020, 'creator': 'PageMaker 7.0', 'page': 18, 'source': 'docs/medical-pharmacology.pdf', 'page_label': '2', 'domain': 'Pharmacology', 'language': 'English'}, page_content='The term ‘drugs’ is being also used to mean\naddictive/abused/illicit substances. However, this\nrestricted and derogatory sense usage is unfor-\ntunate degradation of a time honoured term, and\n‘drug’ should refer to a substance that has some\ntherapeutic/diagnostic application.\nSome other important aspects of pharmacology\nare:\nPharmacotherapeutics It is the application\nof pharmacological information together with\nknowledge of the disease for its prevention,\nmitigation or cure. Selection of the most appro-\npriate drug, dosage and duration of treatment\ntaking into account the specific features of a\np

## Step 5: Create the Retriever

The retriever searches the Chroma vector database and returns the document chunks that are most relevant to the user's question.

In this step, we will retrieve the top five most similar chunks. We are not generating an answer with the language model yet.

In [26]:
# Convert the existing Chroma vector store into a retriever.
#
# search_type="similarity":
# Searches for chunks whose embeddings are most similar
# to the embedding of the user's question.
#
# search_kwargs={"k": 5}:
# Returns the five most relevant chunks.

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

print("Retriever created successfully.")

Retriever created successfully.


## Step 6: Test the Retriever

Before generating answers with the language model, we should verify that the retriever is working correctly.

In this step, we will ask a sample question and retrieve the five most relevant document chunks from the Chroma vector database.

This helps confirm that our embeddings and vector search are working as expected.

In [27]:
# ---------------------------------------------------------
# Test the retriever with a sample question.
#
# The retriever searches the Chroma vector database
# and returns the five most relevant document chunks.
# ---------------------------------------------------------

question = "What is pharmacology?"

retrieved_documents = retriever.invoke(question)

print(f"Question: {question}")
print(f"Number of retrieved documents: {len(retrieved_documents)}")

Question: What is pharmacology?
Number of retrieved documents: 5


## Step 7: Inspect the Retrieved Documents

The retriever returned five document chunks, but we still need to verify that they are relevant to the question.

In this step, we will display the source file, page number, and a short preview of each retrieved chunk.

In [28]:
from pathlib import Path

# Display each retrieved document with its source information.
for index, document in enumerate(retrieved_documents, start=1):

    print("=" * 80)
    print(f"Retrieved Document {index}")

    # Extract the PDF filename from the full source path.
    source_name = Path(
        document.metadata.get("source", "Unknown source")
    ).name

    # PyPDFLoader stores page numbers starting from 0,
    # so we add 1 to display the human-readable page number.
    page_number = document.metadata.get("page")

    if page_number is not None:
        page_number += 1
    else:
        page_number = "Unknown"

    print(f"Source: {source_name}")
    print(f"Page: {page_number}")
    print("-" * 80)

    # Show the first 500 characters of the retrieved chunk.
    print(document.page_content[:500])
    print()

Retrieved Document 1
Source: medical-pharmacology.pdf
Page: 19
--------------------------------------------------------------------------------
The term ‘drugs’ is being also used to mean
addictive/abused/illicit substances. However, this
restricted and derogatory sense usage is unfor-
tunate degradation of a time honoured term, and
‘drug’ should refer to a substance that has some
therapeutic/diagnostic application.
Some other important aspects of pharmacology
are:
Pharmacotherapeutics It is the application
of pharmacological information together with
knowledge of the disease for its prevention,
mitigation or cure. Selection of the most

Retrieved Document 2
Source: medical-pharmacology.pdf
Page: 7
--------------------------------------------------------------------------------
Pharmacology is both a basic and an applied science. It forms the backbone of rational therapeutics.
Whereas the medical student and the prescribing physician are primarily concerned with the applied
aspects, co

## Step 8: Create the Language Model

The retriever can find relevant document chunks, but it cannot write a complete answer.

In this step, we will create a language model using `ChatOpenAI`. Later, the model will receive the retrieved pharmacology content and use it to generate a clear answer.

In [29]:
from langchain_openai import ChatOpenAI

# Create the language model that will generate the final answer.
#
# temperature=0:
# Makes the response more consistent and less creative,
# which is appropriate for factual pharmacology questions.

llm = ChatOpenAI(
    model="gpt-5-nano",
    temperature=0
)

print("Language model created successfully.")

Language model created successfully.


## Step 9: Create the RAG Prompt Template

The prompt defines the instructions that will be sent to the language model.

The model must answer using only the retrieved document context. If the answer is not available in the context, it should clearly say that the information was not found.

In [30]:
from langchain_core.prompts import ChatPromptTemplate

# The system message defines the model's role and rules.
system_message = """
You are PharmacologyGPT, a pharmacology question-answering assistant.

Answer the user's question using only the provided context.

Rules:
1. Do not use information outside the provided context.
2. If the answer is not found in the context, say:
   "I could not find the answer in the provided documents."
3. Give a clear and concise answer.
4. Do not invent drug facts, dosages, warnings, or medical advice.

Context:
{context}
"""

# The human message contains the user's actual question.
human_message = "{input}"

# Create a reusable chat prompt.
rag_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_message),
        ("human", human_message),
    ]
)

print("RAG prompt template created successfully.")

RAG prompt template created successfully.


## Step 10: Create the Document Answering Chain

The document answering chain combines the retrieved document chunks, the RAG prompt, and the language model.

It receives relevant documents as context and asks the language model to generate an answer based only on that context.

## Step 10A: Verify the LangChain Installation

Before creating the document answering chain, verify that the main LangChain package is installed and check its version.

In [32]:
import langchain

print("LangChain version:", langchain.__version__)

LangChain version: 1.3.9


## Step 10B: Verify LangChain Classic

LangChain 1.x moved the older document-combining and retrieval-chain functions into the `langchain-classic` package.

Before creating the document chain, we will verify that this package is available.

In [33]:
import langchain_classic

print("LangChain Classic is installed successfully.")

LangChain Classic is installed successfully.


## Step 10C: Create the Document Answering Chain

The document answering chain combines three components:

1. The retrieved document chunks
2. The RAG prompt template
3. The language model

The chain will receive relevant documents as context and generate an answer based only on that context.

In [34]:
from langchain_classic.chains.combine_documents import (
    create_stuff_documents_chain
)

# Create the document answering chain.
#
# This chain places the retrieved document chunks
# inside the {context} section of the RAG prompt
# and sends the complete prompt to the language model.
document_chain = create_stuff_documents_chain(
    llm=llm,
    prompt=rag_prompt
)

print("Document answering chain created successfully.")

Document answering chain created successfully.


## Step 11: Create the Retrieval Chain

The retrieval chain connects the retriever with the document answering chain.

When the user asks a question, the retriever first finds the most relevant document chunks. Those chunks are then passed to the document answering chain to generate the final response.

In [35]:
from langchain_classic.chains import create_retrieval_chain

# Connect the retriever with the document answering chain.
#
# retriever:
# Finds the most relevant chunks from Chroma.
#
# document_chain:
# Uses those chunks to generate the final answer.

rag_chain = create_retrieval_chain(
    retriever,
    document_chain
)

print("Retrieval chain created successfully.")

Retrieval chain created successfully.


## Step 12: Test the Complete RAG Chain

The complete RAG pipeline is now ready.

In this step, we will send a question to the retrieval chain. The chain will retrieve relevant document chunks from Chroma and use the language model to generate a grounded answer.

In [36]:
# Ask a test question.
question = "What is pharmacology?"

# Run the complete RAG pipeline.
response = rag_chain.invoke(
    {
        "input": question
    }
)

# Display the generated answer.
print("Question:")
print(question)

print("\nAnswer:")
print(response["answer"])

Question:
What is pharmacology?

Answer:
Pharmacology is the science that deals with drugs. It also refers to the interaction between living systems and molecules—especially chemicals introduced from outside the system—and includes knowledge of a drug’s history, source, properties; absorption, biotransformation, distribution, and excretion; mechanism of action; structure–activity relationships; biochemical and physiological effects; and therapeutic uses.


## Step 13: Display the Source Documents

A good RAG system should provide references for its answers.

In this step, we will display the source PDF, page number, and a preview of each retrieved document chunk that was used to generate the answer.

This makes the system more transparent and trustworthy.

In [37]:
from pathlib import Path

# Display the retrieved source documents.
for index, document in enumerate(response["context"], start=1):

    print("=" * 80)
    print(f"Source Document {index}")

    # Display the PDF filename.
    source_name = Path(
        document.metadata.get("source", "Unknown source")
    ).name

    print(f"PDF  : {source_name}")

    # Display the page number (convert from 0-based to 1-based).
    page_number = document.metadata.get("page")

    if page_number is not None:
        page_number += 1
    else:
        page_number = "Unknown"

    print(f"Page : {page_number}")

    print("-" * 80)

    # Display a preview of the retrieved text.
    print(document.page_content[:500])

    print()

Source Document 1
PDF  : medical-pharmacology.pdf
Page : 19
--------------------------------------------------------------------------------
The term ‘drugs’ is being also used to mean
addictive/abused/illicit substances. However, this
restricted and derogatory sense usage is unfor-
tunate degradation of a time honoured term, and
‘drug’ should refer to a substance that has some
therapeutic/diagnostic application.
Some other important aspects of pharmacology
are:
Pharmacotherapeutics It is the application
of pharmacological information together with
knowledge of the disease for its prevention,
mitigation or cure. Selection of the most

Source Document 2
PDF  : medical-pharmacology.pdf
Page : 7
--------------------------------------------------------------------------------
Pharmacology is both a basic and an applied science. It forms the backbone of rational therapeutics.
Whereas the medical student and the prescribing physician are primarily concerned with the applied
aspects, correct 

## Step 14: Add Source Citations to the Answer

A professional RAG application should display the references used to generate an answer.

In this step, we will print the generated answer first and then list the PDF names and page numbers used by the language model.

In [38]:
from pathlib import Path

# Ask a question
question = "What is pharmacology?"

# Run the complete RAG pipeline
response = rag_chain.invoke(
    {
        "input": question
    }
)

# -----------------------------------------------------
# Display the generated answer
# -----------------------------------------------------

print("=" * 80)
print("Answer")
print("=" * 80)

print(response["answer"])

# -----------------------------------------------------
# Display the document sources
# -----------------------------------------------------

print("\n")
print("=" * 80)
print("Sources")
print("=" * 80)

for document in response["context"]:

    source_name = Path(
        document.metadata["source"]
    ).name

    page_number = document.metadata["page"] + 1

    print(f"• {source_name} (Page {page_number})")

Answer
Pharmacology is the science that deals with drugs. More broadly, it is the study of the interaction between living systems and molecules, especially chemicals introduced from outside the system, and it includes the history, sources, properties, absorption, distribution, metabolism, excretion, mechanism of action, effects, and therapeutic uses of drugs.


Sources
• medical-pharmacology.pdf (Page 19)
• medical-pharmacology.pdf (Page 7)
• Pharmacology 7 days.pdf (Page 8)
• Pharmacology-WEB.pdf (Page 27)
• medical-pharmacology.pdf (Page 18)


## Step 15: Remove Duplicate Source Citations

The retriever may return multiple chunks from the same PDF page.

In this step, we will keep only unique combinations of PDF name and page number so the source list is cleaner and easier to read.

In [39]:
from pathlib import Path

# Create an empty set to track unique sources.
unique_sources = set()

print("=" * 80)
print("Sources")
print("=" * 80)

for document in response["context"]:

    # Extract the PDF filename.
    source_name = Path(
        document.metadata.get("source", "Unknown source")
    ).name

    # Convert the page number from 0-based to 1-based.
    page_number = document.metadata.get("page")

    if page_number is not None:
        page_number += 1
    else:
        page_number = "Unknown"

    # Store the PDF name and page number together.
    source_key = (source_name, page_number)

    # Print the source only if it has not already been shown.
    if source_key not in unique_sources:
        unique_sources.add(source_key)

        print(f"• {source_name} (Page {page_number})")

Sources
• medical-pharmacology.pdf (Page 19)
• medical-pharmacology.pdf (Page 7)
• Pharmacology 7 days.pdf (Page 8)
• Pharmacology-WEB.pdf (Page 27)
• medical-pharmacology.pdf (Page 18)


## Step 16: Sort the Source Citations

The source citations are currently displayed in retrieval order.

In this step, we will sort them alphabetically by PDF name and then numerically by page number. This makes the citation list easier to read.

In [40]:
from pathlib import Path

# Store unique source and page combinations.
unique_sources = set()

for document in response["context"]:

    # Get the PDF filename.
    source_name = Path(
        document.metadata.get("source", "Unknown source")
    ).name

    # Convert the page number from 0-based to 1-based.
    page_number = document.metadata.get("page")

    if page_number is not None:
        page_number += 1
    else:
        page_number = 0

    unique_sources.add(
        (source_name, page_number)
    )

# Sort first by PDF name, then by page number.
sorted_sources = sorted(
    unique_sources,
    key=lambda source: (
        source[0].lower(),
        source[1]
    )
)

print("=" * 80)
print("Sources")
print("=" * 80)

for source_name, page_number in sorted_sources:

    if page_number == 0:
        print(f"• {source_name} (Page Unknown)")
    else:
        print(f"• {source_name} (Page {page_number})")

Sources
• medical-pharmacology.pdf (Page 7)
• medical-pharmacology.pdf (Page 18)
• medical-pharmacology.pdf (Page 19)
• Pharmacology 7 days.pdf (Page 8)
• Pharmacology-WEB.pdf (Page 27)


# Step 17: Measure Retrieval Quality

Before improving the retriever, we need to measure its current performance.

A vector database compares the user's question with every document embedding and assigns a similarity score.

By examining these scores, we can understand how confident the retriever is and decide whether additional retrieval techniques are needed.

In [41]:
from pathlib import Path

# ----------------------------------------------------------
# Retrieve documents together with their similarity scores.
#
# Lower scores generally indicate a closer match.
# (The exact meaning depends on the vector store.)
# ----------------------------------------------------------

question = "What is pharmacology?"

results = vectorstore.similarity_search_with_score(
    query=question,
    k=5
)

print("=" * 100)
print(f"Question: {question}")
print("=" * 100)

for index, (document, score) in enumerate(results, start=1):

    source_name = Path(
        document.metadata.get("source", "Unknown source")
    ).name

    page_number = document.metadata.get("page")

    if page_number is not None:
        page_number += 1
    else:
        page_number = "Unknown"

    print(f"\nResult {index}")
    print(f"Score : {score:.4f}")
    print(f"Source: {source_name}")
    print(f"Page  : {page_number}")
    print("-" * 80)

    print(document.page_content[:300])

Question: What is pharmacology?

Result 1
Score : 0.6088
Source: medical-pharmacology.pdf
Page  : 19
--------------------------------------------------------------------------------
The term ‘drugs’ is being also used to mean
addictive/abused/illicit substances. However, this
restricted and derogatory sense usage is unfor-
tunate degradation of a time honoured term, and
‘drug’ should refer to a substance that has some
therapeutic/diagnostic application.
Some other important asp

Result 2
Score : 0.6655
Source: medical-pharmacology.pdf
Page  : 7
--------------------------------------------------------------------------------
Pharmacology is both a basic and an applied science. It forms the backbone of rational therapeutics.
Whereas the medical student and the prescribing physician are primarily concerned with the applied
aspects, correct and skilful application of drugs is impossible without a proper understanding of
th

Result 3
Score : 0.7318
Source: Pharmacology 7 days.pdf
Page  : 8


## Step 18: Improve Retrieval with Maximum Marginal Relevance

Basic similarity search may return several chunks containing very similar information.

Maximum Marginal Relevance, or MMR, balances two goals:

1. Retrieve chunks relevant to the question.
2. Avoid returning overly repetitive chunks.

MMR first finds a larger group of candidate chunks and then selects a smaller, more diverse set.

In [42]:
# Create an MMR retriever.
#
# k=5:
# Return five final document chunks.
#
# fetch_k=20:
# First retrieve twenty possible candidates.
#
# lambda_mult=0.7:
# Give more importance to relevance while still
# encouraging some diversity among the results.

mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 5,
        "fetch_k": 20,
        "lambda_mult": 0.7,
    },
)

print("MMR retriever created successfully.")

MMR retriever created successfully.


## Step 19: Test the MMR Retriever

The MMR retriever is designed to return relevant but less repetitive document chunks.

In this step, we will test it with the same question and inspect the returned sources before connecting it to the RAG chain.

In [43]:
from pathlib import Path

# Use the same question so we can compare
# the MMR results with the original similarity results.
question = "What is pharmacology?"

mmr_documents = mmr_retriever.invoke(question)

print("=" * 100)
print(f"Question: {question}")
print(f"Number of retrieved documents: {len(mmr_documents)}")
print("=" * 100)

for index, document in enumerate(mmr_documents, start=1):

    source_name = Path(
        document.metadata.get("source", "Unknown source")
    ).name

    page_number = document.metadata.get("page")

    if page_number is not None:
        page_number += 1
    else:
        page_number = "Unknown"

    print(f"\nResult {index}")
    print(f"Source: {source_name}")
    print(f"Page  : {page_number}")
    print("-" * 80)

    print(document.page_content[:300])

Question: What is pharmacology?
Number of retrieved documents: 5

Result 1
Source: medical-pharmacology.pdf
Page  : 19
--------------------------------------------------------------------------------
The term ‘drugs’ is being also used to mean
addictive/abused/illicit substances. However, this
restricted and derogatory sense usage is unfor-
tunate degradation of a time honoured term, and
‘drug’ should refer to a substance that has some
therapeutic/diagnostic application.
Some other important asp

Result 2
Source: medical-pharmacology.pdf
Page  : 7
--------------------------------------------------------------------------------
Pharmacology is both a basic and an applied science. It forms the backbone of rational therapeutics.
Whereas the medical student and the prescribing physician are primarily concerned with the applied
aspects, correct and skilful application of drugs is impossible without a proper understanding of
th

Result 3
Source: Pharmacology-WEB.pdf
Page  : 27
--------------

## Step 20: Compare Similarity Search and MMR

Different retrieval strategies work better for different types of questions.

In this step, we will compare the sources returned by standard similarity search and Maximum Marginal Relevance (MMR) using the same question.

This comparison helps us understand when MMR provides additional diversity and when both methods produce similar results.

In [44]:
from pathlib import Path

question = "Explain diabetes treatment."

print("=" * 100)
print("Similarity Search")
print("=" * 100)

similarity_docs = retriever.invoke(question)

for index, document in enumerate(similarity_docs, start=1):

    source_name = Path(document.metadata["source"]).name
    page_number = document.metadata["page"] + 1

    print(f"{index}. {source_name} (Page {page_number})")

print("\n")

print("=" * 100)
print("MMR Search")
print("=" * 100)

mmr_docs = mmr_retriever.invoke(question)

for index, document in enumerate(mmr_docs, start=1):

    source_name = Path(document.metadata["source"]).name
    page_number = document.metadata["page"] + 1

    print(f"{index}. {source_name} (Page {page_number})")

Similarity Search
1. medical-pharmacology.pdf (Page 286)
2. Pharmacology-WEB.pdf (Page 1176)
3. Pharmacology-WEB.pdf (Page 783)
4. Pharmacology-WEB.pdf (Page 1178)
5. Pharmacology-WEB.pdf (Page 806)


MMR Search
1. medical-pharmacology.pdf (Page 286)
2. Pharmacology-WEB.pdf (Page 783)
3. medical-pharmacology.pdf (Page 275)
4. medical-pharmacology.pdf (Page 285)
5. Pharmacology-WEB.pdf (Page 1180)


## Step 21: Add a Similarity Score Threshold

In a production RAG system, we should avoid passing weakly related documents to the language model.

In this step, we will retrieve documents together with their similarity scores and keep only those whose scores are better than a chosen threshold.

This helps reduce irrelevant context and improves answer quality.

In [45]:
from pathlib import Path

# Test question
question = "What is pharmacology?"

# Retrieve more candidates than we need.
results = vectorstore.similarity_search_with_score(
    query=question,
    k=10
)

# ----------------------------------------------------------
# IMPORTANT:
# For Chroma's distance metric,
# LOWER scores indicate better matches.
#
# We will start with a threshold of 0.80.
# We will tune this value later based on experiments.
# ----------------------------------------------------------

distance_threshold = 0.80

filtered_results = []

for document, score in results:

    if score <= distance_threshold:
        filtered_results.append((document, score))

print(f"Retrieved documents : {len(results)}")
print(f"Accepted documents  : {len(filtered_results)}")
print()

for index, (document, score) in enumerate(filtered_results, start=1):

    source = Path(document.metadata["source"]).name
    page = document.metadata["page"] + 1

    print(f"{index}. Score={score:.4f} | {source} | Page {page}")

Retrieved documents : 10
Accepted documents  : 7

1. Score=0.6088 | medical-pharmacology.pdf | Page 19
2. Score=0.6655 | medical-pharmacology.pdf | Page 7
3. Score=0.7318 | Pharmacology 7 days.pdf | Page 8
4. Score=0.7416 | Pharmacology-WEB.pdf | Page 27
5. Score=0.7616 | medical-pharmacology.pdf | Page 18
6. Score=0.7682 | medical-pharmacology.pdf | Page 18
7. Score=0.7783 | Pharmacology-WEB.pdf | Page 46


## Step 22: Create a Threshold-Based Retriever

Instead of always returning a fixed number of documents, we will build a reusable function that filters retrieved documents using a similarity score threshold.

This allows the RAG system to ignore weak matches and provide more reliable context to the language model.

In [46]:
from typing import List
from langchain_core.documents import Document

# ----------------------------------------------------------
# Retrieve only documents that satisfy a distance threshold.
#
# Lower distance = Better semantic match.
# ----------------------------------------------------------

def retrieve_with_threshold(
    question: str,
    max_documents: int = 10,
    distance_threshold: float = 0.80
) -> List[Document]:

    # Retrieve candidate documents with scores.
    results = vectorstore.similarity_search_with_score(
        query=question,
        k=max_documents
    )

    filtered_documents = []

    for document, score in results:

        if score <= distance_threshold:
            filtered_documents.append(document)

    return filtered_documents

print("Threshold-based retriever created successfully.")

Threshold-based retriever created successfully.


## Step 23: Test the Threshold-Based Retriever

The threshold-based retriever has been created, but we should test it before connecting it to the RAG chain.

In this step, we will ask a question and verify how many document chunks pass the distance threshold.

In [47]:
from pathlib import Path

# Test the threshold-based retriever.
question = "What is pharmacology?"

filtered_documents = retrieve_with_threshold(
    question=question,
    max_documents=10,
    distance_threshold=0.80
)

print("=" * 100)
print(f"Question: {question}")
print(f"Accepted documents: {len(filtered_documents)}")
print("=" * 100)

for index, document in enumerate(filtered_documents, start=1):

    source_name = Path(
        document.metadata.get("source", "Unknown source")
    ).name

    page_number = document.metadata.get("page")

    if page_number is not None:
        page_number += 1
    else:
        page_number = "Unknown"

    print(
        f"{index}. {source_name} "
        f"(Page {page_number})"
    )

Question: What is pharmacology?
Accepted documents: 7
1. medical-pharmacology.pdf (Page 19)
2. medical-pharmacology.pdf (Page 7)
3. Pharmacology 7 days.pdf (Page 8)
4. Pharmacology-WEB.pdf (Page 27)
5. medical-pharmacology.pdf (Page 18)
6. medical-pharmacology.pdf (Page 18)
7. Pharmacology-WEB.pdf (Page 46)


# Step 24: Add Conversation Memory

Conversation memory allows the chatbot to remember previous messages.

This enables users to ask follow-up questions naturally without repeating the full context.

In [49]:
from langchain_classic.memory import ConversationBufferMemory

# Create a simple conversation memory.
memory = ConversationBufferMemory(
    return_messages=True
)

print("Conversation memory created successfully.")

Conversation memory created successfully.


/var/folders/ky/lw33txtn4fsdnh2zmn9r1yt80000gn/T/ipykernel_40460/1654713199.py:4: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory(


In [50]:
import langchain_classic.memory

print("Memory module is available.")

Memory module is available.


## Step 24: Add Conversation Memory

Conversation memory stores previous messages so the chatbot can understand simple follow-up questions.

In [51]:
from langchain_classic.memory import ConversationBufferMemory

memory = ConversationBufferMemory(
    return_messages=True
)

print("Conversation memory created successfully.")

Conversation memory created successfully.


# Step 25: Create the Streamlit Application

In this phase, we will build a simple web interface using Streamlit.

Instead of asking questions from a Jupyter Notebook, users will be able to interact with PharmacologyGPT through a browser.


##### app.py file cretaed and run: streamlit app.py

# Step 26: Connect Streamlit to the RAG Pipeline

In this step, we will connect the Streamlit interface to our existing RAG pipeline.

When the user clicks the **Ask** button, the application will send the question to the RAG chain and display the generated answer.

# Step 33: Add a PDF Upload Button

In this step, we will add a file uploader to the Streamlit sidebar.

Users will be able to select a PDF from their computer. In the next steps, we will save the uploaded file and add it to the Chroma vector database.




######## Upload PDF


st.sidebar.markdown("---")

uploaded_file = st.sidebar.file_uploader(
    "📂 Upload a PDF",
    type=["pdf"]
)

if uploaded_file is not None:
    st.sidebar.success(
        f"Uploaded: {uploaded_file.name}"
    )

# Step 34: Save the Uploaded PDF

In this step, we will save the uploaded PDF into the project's `docs` folder.

This makes the file available for indexing and future searches.

# Step 35: Index the Uploaded PDF

In this step, we will read the uploaded PDF, split it into smaller chunks, create embeddings, and add those chunks to the existing Chroma vector database.

After indexing, the uploaded PDF becomes immediately searchable without restarting the application.

In [ ]:
# create pdf_utils.py file

# Step 35: Read and Split the Uploaded PDF

In this step, we will create a reusable function that reads a PDF and splits it into smaller text chunks.

The chunks will be used later to generate embeddings and store them in ChromaDB.

# Step 36 — Add a New PDF to the Vector Database

## Objective

In this step, we add a new PDF to our existing Chroma vector database.

---

## Why do we need this?

When a user uploads a new PDF, the LLM cannot search it directly.

We first need to:

1. Load the PDF
2. Split it into smaller text chunks
3. Generate embeddings
4. Store those embeddings in ChromaDB

Once stored, the document becomes searchable by our RAG application.

---

## Workflow

PDF

↓

Load PDF

↓

Split into Chunks

↓

Generate Embeddings

↓

Store in ChromaDB

↓

Ready for Retrieval

🧪 Test the Function

Before connecting it to Streamlit, let's make sure it works.

Create a temporary file named test_pdf.py in your project folder.

# Step 40 — Rebuild the Vector Database Automatically

Instead of uploading each PDF manually through Streamlit, let's build the database from all PDFs inside the docs/ folder in one step.

This is how most production RAG systems initialize their knowledge base.

build_vector_db.py

Why are we doing this?

This separates knowledge base creation from user uploads.

In a production system:

build_vector_db.py is used once to create the initial knowledge base.
The Streamlit upload feature is used later to add new documents without rebuilding everything.

This is a cleaner architecture and mirrors how enterprise RAG applications are typically organized.

Run python build_vector_db.py and paste the final output. Once that's complete, we'll verify the sidebar statistics and then improve the upload workflow to use the new SHA-256 duplicate detection.